<a href="https://colab.research.google.com/github/idoschw3/Corpus2GeoRAG/blob/main/ApplyNER/NER_fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import pandas as pd
import subprocess
import tempfile
import shutil
import json
import os
import re

IS_COLAB_VM = (
    "google.colab" in sys.modules
    and os.path.exists("/content")
    and os.environ.get("COLAB_RELEASE_TAG") is not None
)

if IS_COLAB_VM:
    from google.colab import userdata
else:
    userdata = None

### Clone Corpus2GeoRAG Repository

In [ ]:
IS_LOCAL = os.name == "nt"

LOCAL_REPO_PATH = Path.home() / "Corpus2GeoRAG"
repo_dir = (
    LOCAL_REPO_PATH
    if IS_LOCAL
    else Path("/content/Corpus2GeoRAG")
)

REPO_PATH = str(repo_dir)

if not repo_dir.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/EtzionR/Corpus2GeoRAG.git",
            REPO_PATH
        ],
        check=True
    )

elif not (repo_dir / ".git").exists():
    raise RuntimeError(
        f"{REPO_PATH} exists but is not a Git checkout."
    )

else:
    # Download remote commit information.
    subprocess.run(
        ["git", "fetch", "origin"],
        cwd=repo_dir,
        check=True
    )

    # Update only if Git can fast-forward safely.
    subprocess.run(
        ["git", "pull", "--ff-only"],
        cwd=repo_dir,
        check=True
    )

os.chdir(REPO_PATH)

print("Runtime:", "Local Windows" if IS_LOCAL else "Colab VM")
print("Working folder:", REPO_PATH)

### Cohere cleaning

this step is to clear Cohere extracted entitiies from refrences.

#### Cohere GT upload and cleaning

Uploading and preparing GT

In [ ]:
# ============================================================
# 1. FILE PATHS
# ============================================================

repo_dir = Path(REPO_PATH)

evaluation_dir = (
    repo_dir
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
)

ground_truth_dir = evaluation_dir / "ground_truth"

cohere_path = ground_truth_dir / "cohere_entities.parquet"
subset_path = evaluation_dir / "evaluation_subset.parquet"
corpus_path = repo_dir / "CollectCorpus" / "DATA.json"

required_files = {
    "Cohere ground truth": cohere_path,
    "Evaluation subset": subset_path,
    "Original Wikipedia corpus": corpus_path
}

for name, path in required_files.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name} not found: {path}")

# ============================================================
# 2. LOAD COHERE GT AND EVALUATION SUBSET
# ============================================================

cohere_gt = pd.read_parquet(cohere_path)
evaluation_df = pd.read_parquet(subset_path)

required_entity_columns = {
    "Source", "Index", "Text",
    "word", "entity_group", "start", "end"
}

required_subset_columns = {"Source", "Index", "Text"}

if not required_entity_columns.issubset(cohere_gt.columns):
    raise ValueError(
        "Missing Cohere columns: "
        f"{required_entity_columns - set(cohere_gt.columns)}"
    )

if not required_subset_columns.issubset(evaluation_df.columns):
    raise ValueError(
        "Missing evaluation columns: "
        f"{required_subset_columns - set(evaluation_df.columns)}"
    )

if evaluation_df.empty:
    raise ValueError("Evaluation subset is empty.")

if evaluation_df[["Source", "Index", "Text"]].isna().any().any():
    raise ValueError("Evaluation subset contains missing identifiers.")

if evaluation_df.duplicated(["Source", "Index"]).any():
    raise ValueError("Duplicate Source/Index pairs in evaluation subset.")

if cohere_gt[["Source", "Index", "Text"]].isna().any().any():
    raise ValueError("Cohere GT contains missing paragraph identifiers.")

# ============================================================
# 3. VERIFY COHERE PARAGRAPH ALIGNMENT
# ============================================================

cohere_paragraphs = cohere_gt[
    ["Source", "Index", "Text"]
].drop_duplicates()

alignment_check = cohere_paragraphs.merge(
    evaluation_df[["Source", "Index", "Text"]],
    on=["Source", "Index", "Text"],
    how="left",
    indicator=True
)

unmatched = alignment_check[
    alignment_check["_merge"] != "both"
]

if not unmatched.empty:
    display(unmatched.head(10))
    raise ValueError(
        f"{len(unmatched)} Cohere paragraphs do not match "
        "the evaluation subset."
    )

# ============================================================
# 4. LOAD ONLY THE RELEVANT WIKIPEDIA PAGES
# ============================================================

with open(corpus_path, "r", encoding="utf-8") as f:
    original_corpus = json.load(f)

if not isinstance(original_corpus, dict):
    raise ValueError("Expected Wikipedia corpus to be a dictionary.")

required_pages = set(evaluation_df["Source"])

missing_pages = required_pages - set(original_corpus)

if missing_pages:
    raise ValueError(
        f"{len(missing_pages)} evaluation pages are missing "
        f"from the original corpus. Examples: "
        f"{sorted(missing_pages)[:10]}"
    )

# Preserve original Wikipedia text, including section headings.
relevant_pages = {
    page: original_corpus[page]
    for page in sorted(required_pages)
}

if not all(isinstance(text, str) for text in relevant_pages.values()):
    raise ValueError("Some Wikipedia pages do not contain text strings.")

del original_corpus

# ============================================================
# 5. VERIFY PARAGRAPH TEXT AGAINST ORIGINAL PAGES
# ============================================================

original_lines = {
    page: set(text.split("\n"))
    for page, text in relevant_pages.items()
}

missing_paragraphs = []

for row in evaluation_df[
    ["Source", "Index", "Text"]
].itertuples(index=False):

    if row.Text not in original_lines[row.Source]:
        missing_paragraphs.append({
            "Source": row.Source,
            "Index": row.Index,
            "Text": row.Text
        })

# ============================================================
# 6. SUMMARY
# ============================================================

print("===== COHERE GT PREPARATION =====")

print(f"Cohere entities loaded: {len(cohere_gt):,}")
print(f"Evaluation paragraphs: {len(evaluation_df):,}")
print(f"Paragraphs with Cohere entities: {len(cohere_paragraphs):,}")
print(f"Relevant Wikipedia pages: {len(relevant_pages):,}")

print("\n===== VALIDATION =====")

print(f"Cohere paragraphs unmatched: {len(unmatched):,}")
print(f"Paragraphs missing from original pages: {len(missing_paragraphs):,}")

if missing_paragraphs:
    print("\nParagraphs requiring inspection:")
    display(pd.DataFrame(missing_paragraphs).head(10))

    raise ValueError(
        "Evaluation paragraphs differ from original Wikipedia text. "
        "Resolve alignment before cleaning."
    )

print("\nAll paragraph alignment checks passed.")
print("Cohere GT and original Wikipedia pages are ready.")

print("\n===== COHERE PREVIEW =====")
display(cohere_gt.head(10))

Cleaning

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

TITLE = "=="
MIN_LENGTH = 5

STOP_HEADINGS = {"references", "external links"}

output_path = (
    Path(REPO_PATH)
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
    / "ground_truth"
    / "cohere_entities_cleaned.parquet"
)

heading_pattern = re.compile(
    r"(?P<marks>={2,})\s*(?P<heading>.*?)\s*(?P=marks)"
)

# ============================================================
# 2. RECONSTRUCT ORIGINAL PARAGRAPH INDICES
# ============================================================

# The evaluation subset was created using the original
# preprocessing. Reproduce that ordering before applying
# the new References cutoff.

original_paragraphs = {}
cutoff_positions = {}
cutoff_headings = {}

for page, page_text in relevant_pages.items():

    kept_paragraphs = []
    first_cutoff = None
    first_heading = None

    for text in page_text.split("\n"):

        heading_match = heading_pattern.fullmatch(text.strip())

        if heading_match:

            heading_level = len(heading_match.group("marks"))
            heading_text = heading_match.group("heading").strip()

            normalized_heading = " ".join(
                heading_text.casefold().split()
            )

            # Record the new cutoff position without changing
            # the original paragraph numbering.
            if (
                first_cutoff is None
                and normalized_heading in STOP_HEADINGS
            ):
                first_cutoff = len(kept_paragraphs)
                first_heading = normalized_heading

            # Original preprocessing stopped at level-2
            # External links only.
            if heading_level == 2 and heading_text == "External links":
                break

            continue

        # Preserve the original paragraph filtering.
        if text.startswith(TITLE) or len(text) < MIN_LENGTH:
            continue

        kept_paragraphs.append(text)

    original_paragraphs[page] = kept_paragraphs

    cutoff_positions[page] = (
        first_cutoff
        if first_cutoff is not None
        else len(kept_paragraphs)
    )

    cutoff_headings[page] = first_heading

# ============================================================
# 3. VALIDATE EVALUATION PARAGRAPH ALIGNMENT
# ============================================================

# Every Source/Index/Text combination must match its
# reconstructed original paragraph before cleaning.

excluded_paragraphs = set()
alignment_errors = []

for row in evaluation_df[
    ["Source", "Index", "Text"]
].itertuples(index=False):

    page = row.Source
    index = int(row.Index)
    text = row.Text

    paragraphs = original_paragraphs[page]

    if (
        index < 0
        or index >= len(paragraphs)
        or paragraphs[index] != text
    ):
        alignment_errors.append({
            "Source": page,
            "Index": index,
            "Text": text
        })
        continue

    # Paragraph lies at or after the new cutoff.
    if index >= cutoff_positions[page]:
        excluded_paragraphs.add((page, index))

if alignment_errors:

    print("Paragraph alignment errors:")
    display(pd.DataFrame(alignment_errors).head(20))

    raise ValueError(
        f"{len(alignment_errors):,} paragraphs could not be "
        "matched to their original indices. "
        "No cleaning output was saved."
    )

print("Original paragraph alignment verified.")

# ============================================================
# 4. IDENTIFY COHERE ENTITIES TO REMOVE
# ============================================================

remove_mask = pd.Series(
    [
        (source, int(index)) in excluded_paragraphs
        for source, index in zip(
            cohere_gt["Source"],
            cohere_gt["Index"]
        )
    ],
    index=cohere_gt.index,
    dtype=bool
)

removed_entities = cohere_gt.loc[remove_mask].copy()

cohere_gt_cleaned = (
    cohere_gt.loc[~remove_mask]
    .copy()
    .reset_index(drop=True)
)

# ============================================================
# 5. VERIFY CLEANING
# ============================================================

original_count = len(cohere_gt)
removed_count = len(removed_entities)
cleaned_count = len(cohere_gt_cleaned)

assert original_count == removed_count + cleaned_count

remaining_keys = set(
    zip(
        cohere_gt_cleaned["Source"],
        cohere_gt_cleaned["Index"].astype(int)
    )
)

assert not (remaining_keys & excluded_paragraphs), (
    "Excluded paragraphs remain in cleaned Cohere GT."
)

# Verify that all remaining entity character offsets
# still match the original paragraph text.

for row in cohere_gt_cleaned[
    ["Text", "word", "start", "end"]
].itertuples(index=False):

    if row.Text[int(row.start):int(row.end)] != row.word:
        raise ValueError(
            "Entity offset validation failed. "
            "No cleaning output was saved."
        )

# ============================================================
# 6. SAVE CLEANED COHERE GT
# ============================================================

output_path.parent.mkdir(parents=True, exist_ok=True)

if output_path.exists():

    existing = pd.read_parquet(output_path)

    if existing.equals(cohere_gt_cleaned):
        print("\nExisting cleaned GT matches. File unchanged.")
    else:
        raise FileExistsError(
            "An existing cleaned GT file differs from "
            "the new results. Inspect it before replacing. "
            f"Path: {output_path}"
        )

else:
    temporary_path = output_path.with_suffix(".tmp")

    cohere_gt_cleaned.to_parquet(
        temporary_path,
        engine="pyarrow",
        compression="zstd",
        index=False
    )

    temporary_path.replace(output_path)

    print("\nCleaned Cohere GT saved successfully.")

# ============================================================
# 7. CLEANING REPORT
# ============================================================

removed_percentage = (
    removed_count / original_count * 100
    if original_count else 0
)

print("\n===== COHERE CLEANING SUMMARY =====")

print(f"Original entities: {original_count:,}")
print(f"Removed entities: {removed_count:,}")
print(f"Cleaned entities: {cleaned_count:,}")
print(f"Entities removed: {removed_percentage:.2f}%")

print(
    "Evaluation paragraphs excluded: "
    f"{len(excluded_paragraphs):,}"
)

print(
    "Pages containing removed entities: "
    f"{removed_entities['Source'].nunique():,}"
)

print("\n===== REMOVED ENTITIES BY CUTOFF =====")

if not removed_entities.empty:

    removed_by_heading = removed_entities["Source"].map(
        cutoff_headings
    ).value_counts()

    for heading, count in removed_by_heading.items():
        print(f"{heading}: {count:,} entities")

    print("\n===== REMOVED ENTITY PREVIEW =====")

    display(
        removed_entities[
            ["Source", "Index", "word", "entity_group"]
        ].head(20)
    )

print("\n===== CLEANED GT PREVIEW =====")
display(cohere_gt_cleaned.head(10))

print(f"\nOutput: {output_path}")

### Commit to Repo

In [ ]:
GITHUB_USER = "idoschw3"
GITHUB_REPO = "Corpus2GeoRAG"
GITHUB_EMAIL = "193781032+idoschw3@users.noreply.github.com"
COMMIT_MESSAGE = "Update NER workflow"

LOCAL_REPO_PATH = Path.home() / GITHUB_REPO
MAX_GIT_FILE_BYTES = 100 * 1024 * 1024

# --------------------------------------------------
# Detect runtime
# --------------------------------------------------

if os.name == "nt":
    IS_LOCAL = True
elif os.environ.get("COLAB_RELEASE_TAG"):
    IS_LOCAL = False
else:
    raise RuntimeError(
        "Could not identify this runtime as Windows or a hosted Colab VM."
    )

if shutil.which("git") is None:
    raise RuntimeError(
        "Git is not available to this Jupyter environment. "
        "Install Git for Windows and restart Jupyter if running locally."
    )

# --------------------------------------------------
# Locate the repository
# --------------------------------------------------

if IS_LOCAL:
    candidates = [Path.cwd(), LOCAL_REPO_PATH]

    previous_repo_path = globals().get("REPO_PATH")
    if previous_repo_path:
        candidates.insert(0, Path(previous_repo_path))

    detected_repo = None

    for candidate in candidates:
        if not candidate.is_dir():
            continue

        probe = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            cwd=candidate,
            text=True,
            encoding="utf-8",
            errors="replace",
            capture_output=True
        )

        if probe.returncode == 0:
            root = Path(probe.stdout.strip()).resolve()

            if root.name.lower() == GITHUB_REPO.lower():
                detected_repo = root
                break

    if detected_repo is None:
        raise FileNotFoundError(
            f"Could not find your local {GITHUB_REPO} checkout.\n"
            f"Clone it to {LOCAL_REPO_PATH}, or update LOCAL_REPO_PATH."
        )

    REPO_PATH = str(detected_repo)

else:
    REPO_PATH = f"/content/{GITHUB_REPO}"

    if not Path(REPO_PATH).is_dir():
        raise FileNotFoundError(
            f"{REPO_PATH} does not exist. Run your clone cell first."
        )

os.chdir(REPO_PATH)

print("Runtime:", "Local Windows" if IS_LOCAL else "Colab VM")
print("Repository:", REPO_PATH)


def run_git(args, *, env=None, show_output=True):
    """Run Git and include its actual output if it fails."""
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_PATH,
        env=env,
        stdin=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True
    )

    if show_output:
        if result.stdout:
            print(result.stdout, end="")
        if result.stderr:
            print(result.stderr, end="")

    if result.returncode != 0:
        details = (result.stderr or result.stdout or "").strip()
        raise RuntimeError(f"git {' '.join(args)} failed:\n{details}")

    return result


# --------------------------------------------------
# Git configuration
# --------------------------------------------------

run_git(["config", "user.name", GITHUB_USER])
run_git(["config", "user.email", GITHUB_EMAIL])
run_git([
    "remote", "set-url", "origin",
    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
])

branch = run_git(
    ["branch", "--show-current"],
    show_output=False
).stdout.strip()

if not branch:
    raise RuntimeError("Could not determine the current Git branch.")

print("Branch:", branch)

# Stop if an earlier rebase or merge is still paused.
for state_path in ("rebase-merge", "rebase-apply", "MERGE_HEAD"):
    git_state = run_git(
        ["rev-parse", "--git-path", state_path],
        show_output=False
    ).stdout.strip()

    if Path(git_state).exists():
        raise RuntimeError(
            "An earlier rebase or merge is still in progress. "
            "Check `git status` and resolve it before rerunning this cell."
        )

# --------------------------------------------------
# Authentication and synchronization
# --------------------------------------------------

git_env = os.environ.copy()
git_env["GIT_TERMINAL_PROMPT"] = "0"

auth_options = []
askpass_dir = None

try:
    if IS_LOCAL:
        # Use Git credentials already configured on your computer.
        print("Authentication: local Git credentials.")

    else:
        from google.colab import userdata

        github_token = userdata.get("GITHUB_TOKEN")

        if not github_token:
            raise ValueError(
                "GITHUB_TOKEN is missing from Colab Secrets. "
                "Add it and enable notebook access."
            )

        askpass_dir = tempfile.TemporaryDirectory(
            prefix="github-askpass-"
        )
        askpass_path = Path(askpass_dir.name) / "askpass.sh"

        askpass_path.write_text(
            "#!/bin/sh\n"
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "$GITHUB_USER" ;;\n'
            '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
            "esac\n",
            encoding="utf-8"
        )
        askpass_path.chmod(0o700)

        git_env["GITHUB_USER"] = GITHUB_USER
        git_env["GITHUB_TOKEN"] = github_token
        git_env["GIT_ASKPASS"] = str(askpass_path)

        auth_options = ["-c", "credential.helper="]

        print("Authentication: Colab GITHUB_TOKEN Secret.")

    # --------------------------------------------------
    # Prevent accidental nested repository
    # --------------------------------------------------

    ignore_entry = f"/{GITHUB_REPO}/"
    gitignore_path = Path(REPO_PATH) / ".gitignore"

    existing_ignore = (
        gitignore_path.read_text(encoding="utf-8")
        if gitignore_path.exists()
        else ""
    )

    if ignore_entry not in existing_ignore.splitlines():
        with gitignore_path.open("a", encoding="utf-8") as f:
            if existing_ignore and not existing_ignore.endswith("\n"):
                f.write("\n")
            f.write(ignore_entry + "\n")

    nested_repo = run_git(
        ["ls-files", "--stage", "--", GITHUB_REPO],
        show_output=False
    ).stdout.strip()

    if nested_repo:
        run_git([
            "rm", "--cached", "-r", "-f", "--", GITHUB_REPO
        ])
        print("Removed the nested repository path from Git tracking.")

    # --------------------------------------------------
    # Find files exceeding GitHub's per-file limit
    # --------------------------------------------------

    candidate_files = run_git(
        [
            "ls-files",
            "--cached",
            "--others",
            "--exclude-standard",
            "-z"
        ],
        show_output=False
    ).stdout.split("\0")

    excluded_files = set()

    for relative_path in sorted(set(candidate_files) - {""}):
        file_path = Path(REPO_PATH) / relative_path

        if not file_path.is_file():
            continue

        file_size = file_path.stat().st_size

        if file_size > MAX_GIT_FILE_BYTES:
            excluded_files.add(relative_path)

            # Remove from Git's index, while keeping the file on disk.
            run_git(
                [
                    "rm",
                    "--cached",
                    "--ignore-unmatch",
                    "-f",
                    "--",
                    f":(top,literal){relative_path}"
                ],
                show_output=False
            )

            print(
                f"Excluded oversized file: {relative_path} "
                f"({file_size / 1024**2:.2f} MiB). "
                "Kept on disk."
            )

    # --------------------------------------------------
    # Stage changes, excluding oversized files
    # --------------------------------------------------

    exclusion_pathspecs = [
        f":(top,literal,exclude){path}"
        for path in sorted(excluded_files)
    ]

    run_git(["add", "-A", "--", ".", *exclusion_pathspecs])

    # Exclude notebook copies, preserving the original cell's behavior.
    staged_files = run_git(
        ["diff", "--cached", "--name-only", "-z"],
        show_output=False
    ).stdout.split("\0")

    for notebook in staged_files:
        if (
            notebook.endswith(".ipynb")
            and notebook not in excluded_files
        ):
            run_git([
                "restore", "--staged", "--",
                f":(top,literal){notebook}"
            ])

    # --------------------------------------------------
    # Create a normal commit
    # --------------------------------------------------

    staged_diff = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        cwd=REPO_PATH,
        stdin=subprocess.DEVNULL
    )

    if staged_diff.returncode == 1:
        run_git(["commit", "-m", COMMIT_MESSAGE])
        print("Repository changes committed.")
    elif staged_diff.returncode == 0:
        print("No new repository changes to commit.")
    else:
        raise RuntimeError("Could not check the staged changes.")

    # --------------------------------------------------
    # Restore tracked notebook copies on disk
    # This discards edits to those on-disk copies.
    # --------------------------------------------------

    tracked_files = run_git(
        ["ls-tree", "-r", "--name-only", "-z", "HEAD"],
        show_output=False
    ).stdout.split("\0")

    for notebook in tracked_files:
        if notebook.endswith(".ipynb"):
            run_git([
                "restore", "--worktree", "--",
                f":(top,literal){notebook}"
            ])

    # --------------------------------------------------
    # Pull/rebase GitHub changes
    # --------------------------------------------------

    pull = subprocess.run(
        ["git", *auth_options, "pull", "--rebase", "origin", branch],
        cwd=REPO_PATH,
        env=git_env,
        stdin=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True
    )

    if pull.stdout:
        print(pull.stdout, end="")
    if pull.stderr:
        print(pull.stderr, end="")

    if pull.returncode != 0:
        conflicted = run_git(
            ["diff", "--name-only", "--diff-filter=U"],
            show_output=False
        ).stdout.splitlines()

        if conflicted == [".gitignore"]:
            ours = run_git(
                ["show", ":2:.gitignore"],
                show_output=False
            ).stdout

            theirs = run_git(
                ["show", ":3:.gitignore"],
                show_output=False
            ).stdout

            if ours.strip() != theirs.strip():
                raise RuntimeError(
                    "The .gitignore versions differ. The rebase is paused; "
                    "inspect the conflict before continuing."
                )

            gitignore_path.write_text(
                ours + ("\n" if ours and not ours.endswith("\n") else ""),
                encoding="utf-8"
            )

            run_git(["add", "--", ".gitignore"])
            run_git([
                "-c", "core.editor=true", "rebase", "--continue"
            ])

        else:
            raise RuntimeError(
                "Pull failed. Check the Git output above and `git status`. "
                "If running locally, verify your GitHub authentication."
            )

    # --------------------------------------------------
    # Push synchronized changes
    # --------------------------------------------------

    run_git(
        [*auth_options, "push", "origin", branch],
        env=git_env
    )

    print(
        f"Successfully pushed to {GITHUB_USER}/{GITHUB_REPO} "
        f"on branch '{branch}'."
    )

finally:
    if askpass_dir is not None:
        askpass_dir.cleanup()

    git_env.pop("GITHUB_TOKEN", None)